[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/04_cell_specific_six_sweep_fitting.ipynb)

## Colab setup and Step 04 run controls

Run the first setup cell before any imports. In Google Colab it clones this repository, installs `requirements.txt` (including Optuna), changes into the repository root, and puts the repo on `sys.path` so `src.*` imports work.

### Selecting the data to fit

By default, the Step 04 run cell targets the full available cell set:

- `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all` -> `selected_file_ids=None`.
- `ASTROMODEL_STEP04_MAX_CELLS=all` -> no cell cap.

For a quick Colab smoke run, set environment variables **before** the Step 04 run cell, for example:

```python
import os
os.environ["ASTROMODEL_STEP04_SELECTED_FILE_IDS"] = "1_DH_1_CONTROL"
os.environ["ASTROMODEL_STEP04_MAX_CELLS"] = "1"
os.environ["ASTROMODEL_STEP04_N_FIT_POINTS"] = "10"
os.environ["ASTROMODEL_STEP04_N_STARTS"] = "1"
os.environ["ASTROMODEL_STEP04_MAX_NFEV_ALL6"] = "2"
os.environ["ASTROMODEL_STEP04_MAX_NFEV_HOLDOUT"] = "1"
```

Step 04 fits the six current sweeps for each selected cell. There is no separate region selector in this notebook; to run a region-specific subset, pass the corresponding comma-separated file IDs through `ASTROMODEL_STEP04_SELECTED_FILE_IDS`. `ASTROMODEL_STEP04_N_FIT_POINTS` controls trace downsampling for speed/accuracy, not which current sweeps are included.

### Optimizer and loss controls

Use these environment variables before the Step 04 run cell:

- `ASTROMODEL_STEP04_OPTIMIZER_BACKEND`: `least_squares` (default), `optuna_scalar`, or `optuna_multi`.
- `ASTROMODEL_STEP04_OPTUNA_N_TRIALS`: number of Optuna trials for Optuna backends.
- `ASTROMODEL_STEP04_OPTUNA_SAMPLER`: `tpe`, `random`, or `nsga2`; multi-objective runs default to NSGA-II when left as `tpe`.
- `ASTROMODEL_STEP04_RUN_HOLDOUT`: `1`/`true`/`yes` to run leave-one-sweep-out validation; set to `0` for fast optimizer smoke tests.
- `ASTROMODEL_STEP04_TRACE_LOSS_TYPE`: `COMBINED` (historical/default), `L2`, `L1`, `HUBER`, or `LOG_COSH`.
- `ASTROMODEL_STEP04_FEATURE_SET`: `primary_no_redundant` (default), `primary`, `primary_plus_slopes`, or `all`.
- `ASTROMODEL_STEP04_TRACE_WEIGHT`, `ASTROMODEL_STEP04_FEATURE_WEIGHT`, and `ASTROMODEL_STEP04_BINARY_WEIGHT`: finite non-negative objective weights.

Limitations: Optuna backends require `optuna` from `requirements.txt`; the deterministic fallback is for tests/smoke use only and is not enabled by this notebook. Step 06 remains validation-only; optimization is performed here in Step 04.

In [ ]:
# Colab / local repository setup
from pathlib import Path
import os, shutil, subprocess, sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

# In Colab, set these before running the notebook if you want Drive persistence:
# os.environ["ASTROMODEL_STEP04_OUTPUT_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_full"
# os.environ["ASTROMODEL_STEP04_BACKUP_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_backups"
# os.environ["ASTROMODEL_STEP04_RUN_LABEL"] = "least_squares_full_2026_05"

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(project_root)])
        except subprocess.CalledProcessError:
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next((c for c in candidates if (c / "src").is_dir() and (c / "data").is_dir()), current)
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Optional Colab/Drive data hook. If the repository clone does not contain
# data/2_K+ Pumps Data, set ASTROMODEL_DATA_DIR to a mounted directory with
# that ATF data before running this cell.
expected_data_dir = project_root / "data" / "2_K+ Pumps Data"
external_data_dir = os.environ.get("ASTROMODEL_DATA_DIR")
if external_data_dir and not expected_data_dir.exists():
    src_data = Path(external_data_dir).expanduser().resolve()
    expected_data_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        expected_data_dir.symlink_to(src_data, target_is_directory=True)
    except OSError:
        shutil.copytree(src_data, expected_data_dir)
if not expected_data_dir.exists():
    raise FileNotFoundError(
        f"Missing Step 04 ATF data at {expected_data_dir}. In Colab, mount Drive "
        "or upload data, then set ASTROMODEL_DATA_DIR to the directory containing the ATF files."
    )

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")

# Step 04 — Cell-specific six-sweep fitting and accepted ensemble construction

This notebook validates Step 04 using the **expected reviewer-facing astrocyte ODE model**.

Scope of this notebook:
- verify that the implemented `src.astro_model.model` matches the expected equations;
- build cell-specific six-sweep fits under that model;
- use Step 02 region-aware thresholds to define accepted ensembles;
- run held-out-sweep screening as part of the reviewer-facing contract.

Claim boundary:
- this notebook creates accepted cell-specific ensembles;
- it does **not** by itself claim biological degeneracy;
- mechanistic decomposition belongs to Step 05;
- predictive robustness beyond held-out sweeps belongs to Step 06.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

from src.astro_model import build_paramdict, model
from src.step04_cell_fits import acceptance_contract_table, load_step02_outputs_or_run

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()

# Notebook-only audit/report outputs are separate from canonical Step 04 outputs.
NOTEBOOK_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_NOTEBOOK_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits_step04_model_aligned_demo',
)).resolve()
STEP04_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits',
)).resolve()
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEP04_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR
PROJECT_ROOT

## Model-alignment audit

In [ ]:
def reference_model(z, t, paramdict):
    Cm_a  = paramdict["Astrocyte"]["Cm_a"]
    g_kir = paramdict["Astrocyte"]["g_kir"]
    A = paramdict["Astrocyte"]["A"]
    g_k_a = paramdict["Astrocyte"]["g_k_a"]
    gl_a = paramdict["Astrocyte"]["gl_a"]
    w_a = paramdict["Astrocyte"]["w_a"]
    K_a0 = paramdict["Astrocyte"]["K_a0"]
    Sig_a = paramdict["Astrocyte"]["Sig_a"]
    gama_t = paramdict["Astrocyte"]["gama_t"]
    gama_s = paramdict["Astrocyte"]["gama_s"]
    Z_th = paramdict["Astrocyte"]["Z_th"]
    Z_s = paramdict["Astrocyte"]["Z_s"]
    Va_0 = paramdict["Astrocyte"]["Va_0"]
    Va_s = paramdict["Astrocyte"]["Va_s"]
    Va_l = paramdict["Astrocyte"]["Va_l"]
    P_k = paramdict["Astrocyte"]["P_k"]
    d_gap = paramdict["Astrocyte"]["d_gap"]
    F = paramdict["Astrocyte"]["F"]
    R = paramdict["Astrocyte"]["R"]
    T = paramdict["Astrocyte"]["T"]
    K_o0 =paramdict["external"]["K_o0"]
    w_o = paramdict["external"]["w_o"]
    epsilon = paramdict["external"]["epsilon"]
    idx = np.where(paramdict["external"]["K_bath"]["time"]<=t)[0][-1]
    K_bath = paramdict["external"]["K_bath"]["value"][idx]
    switching_function = paramdict["Astrocyte"].get("switching_function", "sigmoid")
    if "epsilon_middle" in paramdict["external"] and idx == 1:
      epsilon = epsilon*paramdict["external"]["epsilon_middle"]
    if "w_o_middle" in paramdict["external"] and idx == 1:
      w_o = w_o*paramdict["external"]["w_o_middle"]
    Va  = z[0]
    DK_a_t = z[1]
    K_a_s = z[2]
    Kg = z[3]
    DK_a = DK_a_t + K_a_s
    K_a  = K_a0 +DK_a
    DK_o_a = -(w_a/w_o)*DK_a_t
    K_o  = K_o0 + DK_o_a + Kg
    K_ratio = K_o / K_a
    if K_ratio <= 0: K_ratio = 1e-8
    E_k_a = 25.7 * np.log(K_ratio)
    I_k_a = g_k_a*(Va - E_k_a)
    I_Kir = g_kir * np.sqrt(np.abs(K_o))*(Va - E_k_a)*(1/(1+np.exp((Va - E_k_a)/19.2)))
    PH_a = 0.04*(Va - Va_s)
    P_kgap = d_gap*P_k
    exp_neg_PH_a = np.exp(-PH_a)
    denominator = -1 + np.exp(-PH_a)
    if denominator == 0: denominator = 1e-8
    I_kgap = P_kgap * F * PH_a * (1 / denominator) * ((K_a * exp_neg_PH_a) - K_a0)
    I_l_a  = gl_a*(Va - Va_l)
    if switching_function == "sigmoid":
        Th_s = DK_a / (1 + np.exp((Z_th - DK_a_t) * Z_s))
    elif switching_function == "tanh":
        Th_s = DK_a * (0.5 * (1 + np.tanh((DK_a_t - Z_th) * Z_s)))
    elif switching_function == "hill":
        n = paramdict["Astrocyte"].get("hill_coefficient", 2)
        K_d = paramdict["Astrocyte"].get("K_d", 1)
        Th_s = DK_a * ((DK_a_t ** n) / (K_d ** n + DK_a_t ** n))
    else:
        raise ValueError(f"Unknown switching function type: {switching_function}")
    dVa   = (-1.0/Cm_a)*(I_Kir + I_k_a +I_l_a +I_kgap)
    dDK_a_t = -(gama_t*Sig_a/(w_a*F))*(I_Kir + I_k_a)
    dK_a_s = -Th_s*(gama_s*Sig_a/(w_a*F))* I_kgap
    dKg   =  epsilon*(K_bath-K_o)
    return np.asarray([dVa,dDK_a_t,dK_a_s,dKg], dtype=float)

probe_cases = [
    ('CONTROL', 75, {'gki': 90.0, 'pk': 3e-4, 'd': 0.05, 'gt': 2.0, 'gs': 10.0, 'zth': 70.0, 'zs': 2.5, 'eps': 0.002, 'eps_middle': 1.0, 'wo': 1400.0, 'wo_middle': 1.0, 'ca': 500.0, 'gl_a': 5.0, 'Va_l': -70.0, 'Va_s': -92.0, 'switching_function': 'sigmoid', 'w_a': 2000.0}),
    ('MFA', 125, {'gki': 40.0, 'pk': 5e-5, 'd': 1.5, 'gt': 4.0, 'gs': 22.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 2500.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'tanh', 'w_a': 2000.0}),
    ('MFA_BA', 100, {'gki': 25.0, 'pk': 2e-4, 'd': 1.5, 'gt': 7.0, 'gs': 14.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 1700.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'hill', 'hill_coefficient': 3.0, 'K_d': 1.2, 'w_a': 2000.0}),
]
z = np.array([-80.0, 0.5, 0.2, 0.1], dtype=float)
probe_rows = []
for exp_type, current_na, flat in probe_cases:
    pdict = build_paramdict(exp_type, current_na, flat)
    deltas = []
    for t in [0.0, 11173.0, 12000.0, 21140.0, 22000.0]:
        got = model(z, t, pdict)
        ref = reference_model(z, t, pdict)
        deltas.append(float(np.max(np.abs(got - ref))))
    probe_rows.append({'condition': exp_type, 'current_na': current_na, 'max_abs_rhs_delta': max(deltas), 'status': 'exact_within_float_tolerance' if max(deltas) <= 1e-12 else 'mismatch'})
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / 'model_alignment_probe.csv', index=False)
display(probe_df)
assert (probe_df['status'] == 'exact_within_float_tolerance').all()

## Step 02 contract carried into Step 04

In [ ]:
step02_outputs = load_step02_outputs_or_run(PROJECT_ROOT, reuse_existing=True)
region_counts = step02_outputs['region_condition_cell_counts']
display(region_counts)
contract = acceptance_contract_table()
display(contract)

## Runtime-safe Step 04 fit run

By default this executed-review cell uses a balanced two-cells-per-group hybrid subset. Set `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all` and `ASTROMODEL_STEP04_MAX_CELLS=all` for the full 37-cell Step 04 run.

In [ ]:
import time
import warnings

from scipy.integrate import ODEintWarning
from src.atf_io import load_all_cells
from src.step04_cell_fits import run_step04_cell_specific_six_sweep_fitting
from src.step04_loss import Step04OptimizerConfig, Step04LossConfig, TraceLossConfig
from src.step04_outputs import save_step04_run_snapshot

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ModuleNotFoundError:
    optuna = None

warnings.filterwarnings("ignore", category=ODEintWarning)


def _env_int_or_none(name, default="all"):
    raw = os.environ.get(name, default).strip().lower()
    if raw in {"", "none", "all"}:
        return None
    return int(raw)


def _env_int(name, default):
    return int(os.environ.get(name, str(default)))


def _env_worker_count(name, default):
    raw = os.environ.get(name, str(default)).strip().lower()
    return "auto" if raw in {"", "auto", "all"} else int(raw)


def _select_file_ids_per_group(project_root, per_group):
    if per_group is None:
        return None
    cells = load_all_cells(project_root / "data" / "2_K+ Pumps Data")
    groups = {}
    for cell in cells:
        groups.setdefault((cell.condition, cell.region), []).append(cell.file_id)
    selected = []
    for key in sorted(groups):
        selected.extend(sorted(groups[key])[:per_group])
    return selected


selected_file_ids_raw = os.environ.get("ASTROMODEL_STEP04_SELECTED_FILE_IDS", "group_balanced").strip()
if selected_file_ids_raw.lower() in {"", "none", "all"}:
    selected_file_ids = None
elif selected_file_ids_raw.lower() in {"group_balanced", "per_group", "two_per_group"}:
    selected_file_ids = _select_file_ids_per_group(
        PROJECT_ROOT,
        _env_int_or_none("ASTROMODEL_STEP04_CELLS_PER_GROUP", "2"),
    )
else:
    selected_file_ids = [x.strip() for x in selected_file_ids_raw.split(",") if x.strip()]

# Recommended presets:
# - Smoke test: SELECTED_FILE_IDS="1_DH_1_CONTROL", MAX_CELLS="1", N_FIT_POINTS="8", N_STARTS="1", MAX_NFUNC_EV_ALL6="1", RUN_HOLDOUT="0".
# - Hybrid subset: SELECTED_FILE_IDS="group_balanced", CELLS_PER_GROUP="2", backend="hybrid", OPTUNA_N_TRIALS="50", CELL_WORKERS="auto".
# - Full baseline: SELECTED_FILE_IDS="all", MAX_CELLS="all", backend="least_squares", RUN_HOLDOUT="1".
# - Full hybrid: use backend="hybrid" after the subset run has acceptable runtime and candidate quality.

optimizer_config = Step04OptimizerConfig(
    backend=os.environ.get("ASTROMODEL_STEP04_OPTIMIZER_BACKEND", "hybrid"),
    optuna_n_trials=_env_int("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", "50"),
    optuna_sampler=os.environ.get("ASTROMODEL_STEP04_OPTUNA_SAMPLER", "tpe"),
    optuna_objective=os.environ.get("ASTROMODEL_STEP04_OPTUNA_OBJECTIVE", "metric_scalar"),
    hybrid_scipy_pre_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
    hybrid_scipy_post_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
    hybrid_refine_top_k=_env_int("ASTROMODEL_STEP04_HYBRID_REFINE_TOP_K", "3"),
    candidate_top_k=_env_int("ASTROMODEL_STEP04_CANDIDATE_TOP_K", "500"),
    run_holdout=os.environ.get("ASTROMODEL_STEP04_RUN_HOLDOUT", "1").lower() in {"1", "true", "yes"},
)

loss_config = Step04LossConfig(
    trace=TraceLossConfig(
        loss_type=os.environ.get("ASTROMODEL_STEP04_TRACE_LOSS_TYPE", "COMBINED"),
        gradient_loss_weight=float(os.environ.get("ASTROMODEL_STEP04_GRADIENT_LOSS_WEIGHT", "20.0")),
        delta_huber=float(os.environ.get("ASTROMODEL_STEP04_DELTA_HUBER", "1.0")),
    ),
    feature_set=os.environ.get("ASTROMODEL_STEP04_FEATURE_SET", "primary_no_redundant"),
    trace_weight=float(os.environ.get("ASTROMODEL_STEP04_TRACE_WEIGHT", "1.0")),
    feature_weight=float(os.environ.get("ASTROMODEL_STEP04_FEATURE_WEIGHT", "1.0")),
    binary_weight=float(os.environ.get("ASTROMODEL_STEP04_BINARY_WEIGHT", "1.0")),
)

run_started = time.perf_counter()
results = run_step04_cell_specific_six_sweep_fitting(
    PROJECT_ROOT,
    output_dir=STEP04_OUTPUT_DIR,
    selected_file_ids=selected_file_ids,
    max_cells=_env_int_or_none("ASTROMODEL_STEP04_MAX_CELLS", "all"),
    n_fit_points=_env_int("ASTROMODEL_STEP04_N_FIT_POINTS", "40"),
    n_starts=_env_int("ASTROMODEL_STEP04_N_STARTS", "8"),
    n_fit_scipy_pre_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
    n_fit_optuna_points=_env_int("ASTROMODEL_STEP04_N_FIT_OPTUNA_POINTS", os.environ.get("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", "50")),
    n_fit_scipy_post_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
    max_nfunc_ev_all6=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_ALL6", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_ALL6", "60")),
    max_nfunc_ev_holdout=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_HOLDOUT", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_HOLDOUT", "40")),
    accepted_top_k_per_cell=_env_int("ASTROMODEL_STEP04_ACCEPTED_TOP_K_PER_CELL", "500"),
    loss_config=loss_config,
    optimizer_config=optimizer_config,
    cell_fit_workers=_env_worker_count("ASTROMODEL_STEP04_CELL_WORKERS", "auto"),
)
run_elapsed_s = time.perf_counter() - run_started

summary = results['cell_fit_quality_summary']
accepted = results['accepted_cell_ensembles']
heldout = results['heldout_current_screen']
candidates = results['cell_fit_candidates']
summary.to_csv(OUTPUT_DIR / 'cell_fit_quality_summary.csv', index=False)
accepted.to_csv(OUTPUT_DIR / 'accepted_cell_ensembles.csv', index=False)
heldout.to_csv(OUTPUT_DIR / 'heldout_current_screen.csv', index=False)
candidates.to_csv(OUTPUT_DIR / 'cell_fit_candidates.csv', index=False)
step04_summary_path = STEP04_OUTPUT_DIR / "analysis_summary.json"
step04_summary = json.loads(step04_summary_path.read_text(encoding="utf-8")) if step04_summary_path.exists() else {}
step04_summary.update({
    "notebook_elapsed_s": float(run_elapsed_s),
    "notebook_selected_file_ids": selected_file_ids,
    "notebook_cells_per_group": _env_int_or_none("ASTROMODEL_STEP04_CELLS_PER_GROUP", "2"),
})
(OUTPUT_DIR / 'analysis_summary.json').write_text(json.dumps(step04_summary, indent=2), encoding='utf-8')
snapshot_path = save_step04_run_snapshot(
    STEP04_OUTPUT_DIR,
    backup_dir=os.environ.get("ASTROMODEL_STEP04_BACKUP_DIR"),
    label=os.environ.get("ASTROMODEL_STEP04_RUN_LABEL"),
)
print(f"Step 04 elapsed: {run_elapsed_s:.2f} s")
print(f"Selected cells: {len(selected_file_ids) if selected_file_ids is not None else 'all'}")
print(f"Snapshot: {snapshot_path}")
print(f"SQLite DB: {STEP04_OUTPUT_DIR / 'step04_cell_fits.sqlite'}")
display(summary)


## Accepted candidates, held-out screen, and cross-condition audit context

In [ ]:
if accepted.empty:
    print('No accepted candidates for this run.')
else:
    display(accepted[['file_id','condition','region','candidate_id','mean_trace_rmse_mV','mean_weighted_pass_fraction','accepted_all6']])
if heldout.empty:
    print('Held-out screen is empty or disabled for this run.')
else:
    display(heldout[['file_id','heldout_sweep','heldout_trace_rmse_mV','heldout_weighted_pass_fraction','heldout_pass']])

extra_summaries = []
for path in [
    PROJECT_ROOT / 'outputs' / 'bench_DH_1_MFA' / 'cell_fit_quality_summary.csv',
    PROJECT_ROOT / 'outputs' / 'bench_DH_1_MFA_BA' / 'cell_fit_quality_summary.csv',
]:
    if path.exists():
        extra_summaries.append(pd.read_csv(path))
if extra_summaries:
    extra_df = pd.concat(extra_summaries, ignore_index=True)
    extra_df.to_csv(OUTPUT_DIR / 'condition_audit_summary.csv', index=False)
    display(extra_df[['file_id','condition','region','best_trace_rmse_mV','best_weighted_pass_fraction','holdout_pass_count','cell_reviewer_facing']])

## Interpretation boundary

This notebook demonstrates that Step 04 now fits the **expected reviewer-facing model** and produces cell-specific accepted ensembles under a six-sweep contract.

It does **not** by itself establish biological degeneracy.

What it supports:
- the fitted model is the expected ODE model discussed with reviewers;
- Step 04 uses one shared cell-level mechanism across six sweeps;
- held-out-sweep screening is part of the acceptance contract.

What remains for later steps:
- Step 05: mechanistic decomposition of accepted ensembles;
- Step 06: broader predictive robustness and perturbation validation;
- Step 07+: assumption sensitivity and parameter plausibility layers.